# ECE1512 — Project B, Part 2: Learned Image Compression

Complete the marked `TODO` sections and run the notebook in order. Use the validation split to select model settings; evaluate the held-out test split only after selection.


**SSIM protocol:** RGB tensors in `[0,1]`, TorchMetrics Gaussian window 11×11, sigma 1.5, `data_range=1`. Compute one SSIM per image and average equally, matching the restoration project. SSIM, PSNR, and file bpp describe different properties; report all three. Historical executed notebooks retain their original outputs until rerun.


## Step 0 — Setup and data

Install dependencies if needed:

```bash
python -m pip install torch torchmetrics numpy pillow matplotlib notebook
```

Doenload  images from 
https://huggingface.co/datasets/LeonYe/ADPPatches

Place the `data_1um_bicubic` image folder in the parent directory of this notebook, as in the original project, or specify a custom path using `DATA_ROOT` below.

In [ ]:
from pathlib import Path
import sys, json, copy, math, io
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'instructor':
    PROJECT_DIR = PROJECT_DIR.parent
if (PROJECT_DIR / 'dnn_compression_notebook').is_dir():
    PROJECT_DIR = PROJECT_DIR / 'dnn_compression_notebook'
assert (PROJECT_DIR / 'model.py').is_file(), 'Set PROJECT_DIR to the helper folder'
sys.path.insert(0, str(PROJECT_DIR))
DATA_ROOT = PROJECT_DIR.parent / 'data_1um_bicubic'
OUTPUT_ROOT = PROJECT_DIR / 'runs'
from data import seed_everything, split_images, make_loaders
from model import CompressionModel
from engine import fit, evaluate, load_checkpoint
from checks import check_soft_quantize, check_pipeline
from codec import compress, decompress
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    torch.set_num_threads(min(4, torch.get_num_threads()))
print('Device:', DEVICE, '| PyTorch:', torch.__version__)

## 2. Configuration

In [ ]:
MODE = 'larger'
SEED = 1512
DATA_CONFIG = dict(crop_size=64, batch_size=32)
MODEL_CONFIG = {'width': 64, 'latent_dim': 16, 'levels': 1, 'codebook_size': 256, 'straight_through': True}
TRAIN_CONFIG = dict(epochs=40, lr=2e-4, rate_weight=1e-4,
                    temperature_start=0.05, temperature_end=0.01)
LIMITS = dict(train=1024, val=128, test=128)
seed_everything(SEED)
SPLIT_MANIFEST = OUTPUT_ROOT / 'shared_data_splits.json'
splits = split_images(DATA_ROOT, SPLIT_MANIFEST, seed=SEED)
loaders = make_loaders(splits, seed=SEED, limits=LIMITS, **DATA_CONFIG)
from data import ImageDataset
from torch.utils.data import DataLoader
all_test_paths = make_loaders(splits, seed=SEED, limits=None, **DATA_CONFIG)['test'].dataset.paths
loaders['test'] = DataLoader(ImageDataset(all_test_paths[128:384], crop_size=64, train=False),
                            batch_size=32, shuffle=False)
print('Fixed held-out test crops:', len(loaders['test'].dataset))
print('Images used for training/validation:', {s: len(loaders[s].dataset) for s in ('train','val')})
x = next(iter(loaders['train']))
print('Input:', tuple(x.shape))
fig, axes = plt.subplots(1,4,figsize=(10,3))
for ax, image in zip(axes,x[:4]):
    ax.imshow(image.permute(1,2,0)); ax.axis('off')
plt.show()


## Step 2 — Implement soft vector quantization

The encoder returns `z: [B,D,h,w]`. The helper rearranges it to `flat: [N,D]`, where `N=Bhw`. The trainable codebook is `codebook: [K,D]`.

For each latent vector $z_i$ and codeword $e_k$:

$$d_{ik}=\|z_i-e_k\|_2^2=\|z_i\|^2+\|e_k\|^2-2z_i^Te_k,$$
$$p_{ik}=\frac{\exp(-d_{ik}/\tau)}{\sum_j\exp(-d_{ij}/\tau)},\qquad \widetilde z_i=\sum_k p_{ik} e_k.$$

Complete TODO 2A–2C below. Use matrix multiplication for distances to avoid allocating `[N,K,D]`. Normalize across **codewords**, the last dimension. Use a numerically stable softmax. Do not detach, convert to NumPy, round, or take argmin in this soft path. Return `(soft_vectors, probabilities)`.

**Questions:** Why does smaller temperature approach nearest-neighbor quantization? What happens for large temperature? Why is a weighted average of codeword *indices* not the same as a weighted average of codeword *vectors*?

In [ ]:
def TODO(hint):
    raise NotImplementedError('Complete the student exercise: ' + hint)

def soft_quantize(flat, codebook, temperature):
    """
    Args:
        flat: [N, D] latent vectors
        codebook: [K, D] codewords
        temperature: positive scalar

    Returns:
        soft_vectors: [N, D]
        probabilities: [N, K]
    """
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    # TODO 1: Compute squared Euclidean distances [N, K].
    distances = None

    # TODO 2: Compute soft assignment probabilities [N, K].
    probabilities = None

    # TODO 3: Compute weighted codebook vectors [N, D].
    soft_vectors = None
    
    return soft_vectors, probabilities

## Step 3 — Check values, temperature behavior, and gradients

These checks test known distances, normalization, cold/hot limits, finite-difference gradients, and the complete model. A shape check alone is insufficient: a detached tensor can have the correct shape while preventing the encoder from learning.

During evaluation, the helper selects the closest codeword and decodes its vector. It never transmits the soft mixture. A test also encodes and decodes a non-square image whose dimensions are not divisible by the downsampling factor.

In [ ]:
check_soft_quantize(soft_quantize)
check_pipeline(soft_quantize)
seed_everything(SEED)
model = CompressionModel(soft_quantize, **MODEL_CONFIG).to(DEVICE)
print('Trainable parameters:', f'{sum(p.numel() for p in model.parameters()):,}')
with torch.no_grad():
    latent = model.encode(x[:1].to(DEVICE))
print('Latent shape:', tuple(latent.shape))
print('Assignment matrix elements per batch:', DATA_CONFIG['batch_size'] * latent.shape[-2] * latent.shape[-1] * MODEL_CONFIG['codebook_size'])

## PSNR and SSIM implementations

Compute one value per image for RGB tensors in `[0,1]`. These functions are used throughout validation, testing, JPEG comparison, and visualization.


In [ ]:
import torch.nn.functional as F
def _metric_inputs(pred, target):
    """Validate matching [B,3,H,W] RGB tensors and compute in float32."""
    if pred.shape != target.shape or pred.ndim != 4 or pred.shape[1] != 3:
        raise ValueError('Expected matching [B,3,H,W] RGB tensors')
    if pred.device != target.device:
        raise ValueError('Prediction and target must be on the same device')
    return pred.float(), target.float()


def psnr_per_image(pred, target):
    """Return [B] PSNR in dB for RGB tensors with data range [0,1].

    Average squared error over channels and pixels independently per image.
    The 1e-12 MSE floor gives 120 dB for identical images.
    """
    # TODO
    return 0


def ssim_per_image(pred, target):
    """Return [B] RGB SSIM: Gaussian 11x11, sigma 1.5, data_range=1.

    Population moments are computed independently per channel. Exclude the
    five-pixel border before averaging, matching the TorchMetrics protocol.
    """
    

    pred, target = _metric_inputs(pred, target)
    if min(pred.shape[-2:]) < 11:
        raise ValueError('SSIM requires images at least 11x11 pixels')
    # TODO
    return 0


In [ ]:
# Reference functions are used only to check your implementation.
from metrics import ssim_per_image as reference_ssim
from functools import partial
import engine as training_engine

metric_generator = torch.Generator().manual_seed(1512)
metric_target = torch.rand(2, 3, 24, 28, generator=metric_generator, device='cpu').to(DEVICE)
metric_pred = (0.8 * metric_target + 0.1).clamp(0, 1)
expected_mse = (metric_pred - metric_target).square().flatten(1).mean(1)
for pred, target in [(metric_target, metric_target), (metric_pred, metric_target),
                     (torch.zeros_like(metric_target), torch.ones_like(metric_target))]:
    actual_psnr = psnr_per_image(pred, target)
    actual_ssim = ssim_per_image(pred, target)
    assert actual_psnr.shape == actual_ssim.shape == (2,)
    assert actual_psnr.device == actual_ssim.device == target.device
    expected_psnr = -10 * (pred - target).square().flatten(1).mean(1).clamp_min(1e-12).log10()
    torch.testing.assert_close(actual_psnr, expected_psnr)
    torch.testing.assert_close(actual_ssim, reference_ssim(pred, target), rtol=1e-4, atol=1e-5)
# Different errors in one batch catch accidentally averaging before log10.
mixed_pred = torch.stack([metric_target[0], metric_pred[1]])
torch.testing.assert_close(psnr_per_image(mixed_pred, metric_target),
                          torch.stack([expected_mse.new_tensor(120.), -10 * expected_mse[1].log10()]))

# Bind the student implementations to all evaluation and training calls below.
student_metrics = dict(psnr_fn=psnr_per_image, ssim_fn=ssim_per_image)
evaluate = partial(training_engine.evaluate, **student_metrics)
fit = partial(training_engine.fit, **student_metrics)

## Step 4 — Understand the provided end-to-end objective

The learned categorical prior is $r_k=\mathrm{softmax}(a)_k$. With $H,W$ measured in input pixels:

$$R_{soft}=\frac{\sum_{i,k}p_{ik}[-\log_2 r_k]}{BHW},\qquad L=\mathrm{MSE}(x,\widehat x)+\lambda R_{soft}.$$

All encoder, decoder, codebook, and prior parameters are optimized jointly with Adam. MSE is measured on `[0,1]` RGB; its scale matters when choosing $\lambda$. The temperature decreases geometrically from the configured start to end value. Gradient clipping limits the total norm to 1.

At validation/test time, $R_{hard}=\sum_i-\log_2 r_{k_i}/(BHW)$ uses hard nearest-neighbor indices. We choose the checkpoint with the lowest **hard validation** `MSE + rate_weight × estimated_bpp`. PSNR is computed per image and then averaged. Neither this rate estimate nor codebook entropy is a measured file size.

**Questions:** Which parameters receive gradients from reconstruction and rate? Why can soft-training quality differ from hard-evaluation quality? Why does increasing the rate penalty tend to trade fidelity for predictability? What does a collapsed codebook look like?

## Step 5 — Train the model


In [ ]:
from datetime import datetime
import csv
sys.path.insert(0, str(PROJECT_DIR / 'instructor'))
from warm_start import warm_start
USE_WARM_START = True
REUSE_CHECKPOINT = None  # Optional path to a completed run's best.pt; None trains from scratch.
seed_everything(SEED)
loaders = make_loaders(splits, seed=SEED, limits=LIMITS, **DATA_CONFIG)
from data import ImageDataset
from torch.utils.data import DataLoader
all_test_paths = make_loaders(splits, seed=SEED, limits=None, **DATA_CONFIG)['test'].dataset.paths
loaders['test'] = DataLoader(ImageDataset(all_test_paths[128:384], crop_size=64, train=False),
                            batch_size=32, shuffle=False)
print('Fixed held-out test crops:', len(loaders['test'].dataset))
if REUSE_CHECKPOINT is None:
    RUN_DIR = OUTPUT_ROOT / ('solution_' + MODE + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    model = CompressionModel(soft_quantize, **MODEL_CONFIG).to(DEVICE)
    initialization = None
    if USE_WARM_START:
        initialization = warm_start(model, loaders['train'], DEVICE, epochs=8, max_vectors=32768)
        TRAIN_CONFIG.update(temperature_start=initialization['temperature_start'],
                            temperature_end=initialization['temperature_start'] * 0.2)
        print('Training-only initialization:', initialization)
    model, history = fit(model, loaders, TRAIN_CONFIG, RUN_DIR, DEVICE)
    if initialization is not None:
        (RUN_DIR / 'warmup.json').write_text(json.dumps(initialization, indent=2))
else:
    RUN_DIR = Path(REUSE_CHECKPOINT).resolve().parent
    model, saved_checkpoint = load_checkpoint(REUSE_CHECKPOINT, soft_quantize, DEVICE)
    assert model.config == MODEL_CONFIG, 'Reuse a checkpoint with the displayed architecture'
    TRAIN_CONFIG = saved_checkpoint['train_config']
    with (RUN_DIR / 'history.csv').open() as handle:
        history = [{k: float(v) for k,v in row.items()} for row in csv.DictReader(handle)]
    print('Loaded completed training run:', REUSE_CHECKPOINT)
    if (RUN_DIR / 'warmup.json').exists():
        print('Training-only initialization:', (RUN_DIR / 'warmup.json').read_text())
    print('Best validation:', saved_checkpoint['validation'])
protocol = dict(seed=SEED, mode=MODE, data_root=str(DATA_ROOT.resolve()),
                manifest=str(SPLIT_MANIFEST.resolve()), data_config=DATA_CONFIG, limits=LIMITS,
                used_paths={s: [str(p.relative_to(DATA_ROOT)) for p in loader.dataset.paths]
                            for s, loader in loaders.items()})
(RUN_DIR / 'data_protocol.json').write_text(json.dumps(protocol, indent=2))
print('Saved:', RUN_DIR)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
epochs = [r['epoch'] for r in history]
axes[0].plot(epochs, [r['train_loss'] for r in history], label='soft train')
axes[0].plot(epochs, [r['val_loss'] for r in history], label='hard validation')
axes[0].set_ylabel('Rate-distortion loss'); axes[0].legend()
axes[1].plot(epochs, [r['val_psnr'] for r in history]); axes[1].set_ylabel('Hard validation PSNR (dB)')
axes[2].plot(epochs, [r['val_estimated_bpp'] for r in history]); axes[2].set_ylabel('Estimated validation bpp')
for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(RUN_DIR / 'learning_curves.png', dpi=160); plt.show()
if all("val_ssim" in row for row in history):
    plt.figure(figsize=(6,3))
    plt.plot(epochs, [row["val_ssim"] for row in history])
    plt.xlabel("Epoch"); plt.ylabel("Hard validation RGB SSIM"); plt.grid(alpha=.2)
    plt.tight_layout(); plt.savefig(RUN_DIR / "validation_ssim.png", dpi=160); plt.show()


## Step 6 — Tune on validation data

Change one factor at a time: temperature end (`0.02, 0.05, 0.2`), rate weight (`0, 0.001, 0.01`), codebook size (`32, 64, 128`), width (`16, 24, 32`), latent dimension (`4, 8, 16`), or levels (`2, 3`). Keep the split, crop size, sample budget, seed, and epoch budget fixed within a comparison.

Optional `straight_through=True` uses hard vectors in the forward pass and soft-mixture gradients in the backward pass. Compare this to the default fully soft training path as a research extension.

Run at least three configurations for your report. The optional cell below runs rate-weight experiments and plots validation PSNR against **estimated** rate. Choose the desired rate/quality tradeoff from validation; losses with different rate weights are not directly comparable. Each fit uses its own rate weight for epoch selection.

In [ ]:
RUN_SWEEP = False  # Enable after the baseline works; costs three training runs.
validation_runs = []
if RUN_SWEEP:
    for rate_weight in (0.0, 0.001, 0.01):
        seed_everything(SEED)
        candidate_config = dict(TRAIN_CONFIG, rate_weight=rate_weight)
        candidate_loaders = make_loaders(splits, seed=SEED, limits=LIMITS, **DATA_CONFIG)
        candidate = CompressionModel(soft_quantize, **MODEL_CONFIG).to(DEVICE)
        destination = OUTPUT_ROOT / ('sweep_' + str(rate_weight) + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        if USE_WARM_START:
            init = warm_start(candidate, candidate_loaders['train'], DEVICE, epochs=8, max_vectors=32768)
            candidate_config.update(temperature_start=init['temperature_start'], temperature_end=init['temperature_start']*.2)
        candidate, _ = fit(candidate, candidate_loaders, candidate_config, destination, DEVICE)
        (destination / 'data_protocol.json').write_text(json.dumps(protocol, indent=2))
        metrics = evaluate(candidate, candidate_loaders['val'], DEVICE, rate_weight)
        validation_runs.append(dict(rate_weight=rate_weight, checkpoint=str(destination/'best.pt'), **metrics))
        del candidate
    print(json.dumps(validation_runs, indent=2))
    (RUN_DIR / 'validation_sweep.json').write_text(json.dumps(validation_runs, indent=2))
    plt.scatter([r['estimated_bpp'] for r in validation_runs], [r['psnr'] for r in validation_runs])
    for r in validation_runs: plt.annotate(str(r['rate_weight']), (r['estimated_bpp'], r['psnr']))
    plt.xlabel('Estimated validation bpp'); plt.ylabel('Validation PSNR (dB)'); plt.show()

## Step 7 — Evaluate on the test set

Select a checkpoint using validation results only, then run this section. By default it evaluates the baseline. 

In [ ]:
SELECTED_CHECKPOINT = RUN_DIR / 'best.pt'  # Replace with a validation-selected sweep checkpoint if needed.
model, checkpoint = load_checkpoint(SELECTED_CHECKPOINT, soft_quantize, DEVICE)
test_metrics = evaluate(model, loaders['test'], DEVICE, checkpoint['train_config']['rate_weight'])
print('Selected epoch:', checkpoint['epoch'])
print(json.dumps(test_metrics, indent=2))
(RUN_DIR / 'test_metrics.json').write_text(json.dumps(dict(checkpoint=str(SELECTED_CHECKPOINT), **test_metrics), indent=2))
images = next(iter(loaders['test']))[:4].to(DEVICE)
with torch.no_grad():
    result = model(images)
fig, axes = plt.subplots(2, len(images), figsize=(3*len(images), 6), squeeze=False)
for i in range(len(images)):
    axes[0, i].imshow(images[i].cpu().permute(1,2,0)); axes[0, i].set_title('Original')
    axes[1, i].imshow(result['reconstruction'][i].cpu().permute(1,2,0)); axes[1, i].set_title('Hard reconstruction')
    axes[0, i].axis('off'); axes[1, i].axis('off')
plt.tight_layout(); plt.savefig(RUN_DIR / 'reconstructions.png'); plt.show()
counts = torch.bincount(result['indices'].flatten(), minlength=model.config['codebook_size']).cpu()
plt.bar(range(len(counts)), counts); plt.xlabel('Codeword'); plt.ylabel('Count (displayed test batch)'); plt.show()
print('Used codewords in displayed batch:', int((counts > 0).sum()), '/', len(counts))

## Step 8 — Save a compressed file and compare actual sizes with JPEG

In [ ]:
import comparison
compare_jpeg = partial(comparison.compare_jpeg, **student_metrics)
visualize_decoded = partial(comparison.visualize_decoded, **student_metrics)
COMPARISON_DIR = RUN_DIR / 'jpeg_comparison'
comparison_summary, comparison_examples = compare_jpeg(
    model, loaders['test'], DEVICE, COMPARISON_DIR,
    qualities=(1, 10, 20, 30, 50, 70, 90), examples=4,
)
print('Shared checkpoint bytes (excluded from bpp):', Path(SELECTED_CHECKPOINT).stat().st_size)


## Optional dense JPEG comparison

Set `RUN_DENSE_JPEG = True` to recompute JPEG qualities 1–95 using your PSNR and SSIM implementations on the selected test loader. This is evaluation after model selection.


In [ ]:
RUN_DENSE_JPEG = False  # Optional: expensive evaluation over all 95 qualities.
if RUN_DENSE_JPEG:
    dense_summary, _ = compare_jpeg(
        model, loaders['test'], DEVICE, RUN_DIR / 'dense_jpeg_comparison',
        qualities=tuple(range(1, 96)), examples=0,
    )


## Final visualization — the decoded image
Decode the actual saved DNN bitstream, and display it beside the original and JPEG Q10. Labels show each actual file bitrate and PSNR; error maps share a color scale.


In [ ]:
decoder_model, _ = load_checkpoint(SELECTED_CHECKPOINT, soft_quantize, DEVICE)
decoded_image = visualize_decoded(
    decoder_model, COMPARISON_DIR / 'image_0.ecvq',
    COMPARISON_DIR / 'image_0_original.png', COMPARISON_DIR, jpeg_quality=10,
)
